# ANALYSIS OF snoverATAC-seq sample RL3295 - peaks called with MACS2 NFR peaks within same sample

## Pre-processing

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import episcanpy.api as epi
from os import listdir

In [2]:
sc.settings.verbosity = 3 # shows errors (0), warnings (1), info (2), hints (3), progress (4) - 4 useful for debugging
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80,facecolor='None')
sns.set_context('paper')

scanpy==1.9.1 anndata==0.8.0 umap==0.5.3 numpy==1.21.6 scipy==1.7.3 pandas==1.3.5 scikit-learn==1.0.2 statsmodels==0.13.2 python-igraph==0.9.10 pynndescent==0.5.6


## Load the data

In [3]:
cellranger_path = '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/'

In [4]:
samples = listdir(cellranger_path)

In [5]:
samples

['RL3295_21_dirty_THS_37_shortREV_highTn5',
 'RL3295_39_dirty_THS_30_blockedREV_highTn5',
 'RL3295_40_dirty_OMNI_30_blockedREV_lowTn5',
 'RL3295_23_dirty_OMNI_37_shortREV_mediumTn5',
 'RL3295_25_dirty_THS_30_blockedREV_lowTn5',
 'RL3295_72_sucrose_OMNI_37_shortREV_highTn5',
 'RL3295_60_sucrose_OMNI_37_shortREV_highTn5',
 'RL3295_33_dirty_THS_37_blockedREV_highTn5',
 'RL3295_36_dirty_OMNI_37_blockedREV_highTn5',
 'RL3295_94_sucrose_OMNI_37_blockedREV_lowTn5',
 'RL3295_77_sucrose_OMNI_30_blockedREV_mediumTn5',
 'RL3295_57_sucrose_THS_37_shortREV_highTn5',
 'RL3295_27_dirty_THS_30_blockedREV_highTn5',
 'RL3295_38_dirty_THS_30_blockedREV_mediumTn5',
 'RL3295_76_sucrose_OMNI_30_blockedREV_lowTn5',
 'RL3295_42_dirty_OMNI_30_blockedREV_highTn5',
 'RL3295_34_dirty_OMNI_37_blockedREV_lowTn5',
 'RL3295_06_dirty_OMNI_30_shortREV_highTn5',
 'RL3295_07_dirty_THS_37_shortREV_lowTn5',
 'RL3295_08_dirty_THS_37_shortREV_mediumTn5',
 'RL3295_10_dirty_OMNI_37_shortREV_lowTn5',
 'RL3295_09_dirty_THS_37_sh

In [6]:
filepaths = [cellranger_path + sample + '/outs/raw_peak_bc_matrix.h5' for sample in samples]
matrixpaths = [cellranger_path + sample + '/outs/raw_peak_bc_matrix/matrix.mtx' for sample in samples]
cellpaths = [cellranger_path + sample + '/outs/raw_peak_bc_matrix/barcodes.tsv' for sample in samples]
featurepaths = [cellranger_path + sample + '/outs/raw_peak_bc_matrix/peaks.bed' for sample in samples]

In [7]:
filepaths

['/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_21_dirty_THS_37_shortREV_highTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_39_dirty_THS_30_blockedREV_highTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_40_dirty_OMNI_30_blockedREV_lowTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_23_dirty_OMNI_37_shortREV_mediumTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_25_dirty_THS_30_blockedREV_lowTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3295/RL3295_cellrangerOnMACS2peaks/RL3295_72_sucrose_OMNI_37_shortREV_highTn5/outs/raw_peak_bc_matrix.h5',
 '/home/dmakosa/working_data_02/snoverATAC/RL3294_RL3

In [8]:
sampleRL = [sample.split("_")[0] for sample in samples]
sampleWell = [sample.split("_")[1] for sample in samples]
samplePurity = [sample.split("_")[2] for sample in samples]
sampleTagBuffer = [sample.split("_")[3] for sample in samples]
sampleTagTemp = [sample.split("_")[4] for sample in samples]
sampleMErev = [sample.split("_")[5] for sample in samples]
sampleTn5conc = [sample.split("_")[6] for sample in samples]

In [9]:
import copy
adatas = copy.deepcopy(samples)
for i,item in enumerate(matrixpaths):
    adatas[i] = epi.pp.read_ATAC_10x(matrixpaths[i], cell_names = cellpaths[i], var_names = featurepaths[i])

In [10]:
[item.shape for item in adatas]

[(95151, 307084),
 (94615, 307084),
 (76432, 307084),
 (87633, 307084),
 (76499, 307084),
 (78811, 307084),
 (79568, 307084),
 (113255, 307084),
 (119609, 307084),
 (71341, 307084),
 (64413, 307084),
 (87018, 307084),
 (105162, 307084),
 (94204, 307084),
 (57189, 307084),
 (93389, 307084),
 (69091, 307084),
 (97380, 307084),
 (64281, 307084),
 (69866, 307084),
 (72476, 307084),
 (77516, 307084),
 (78024, 307084),
 (47754, 307084),
 (96266, 307084),
 (47558, 307084),
 (55852, 307084),
 (67170, 307084),
 (57018, 307084),
 (68984, 307084),
 (84476, 307084),
 (52496, 307084),
 (78280, 307084),
 (76100, 307084),
 (93479, 307084),
 (71355, 307084),
 (102002, 307084),
 (108716, 307084),
 (25731, 307084),
 (70290, 307084),
 (61688, 307084),
 (78536, 307084),
 (74413, 307084),
 (77592, 307084),
 (51122, 307084),
 (83160, 307084),
 (83098, 307084),
 (83747, 307084),
 (99994, 307084),
 (108578, 307084),
 (82542, 307084),
 (98824, 307084),
 (37475, 307084),
 (81995, 307084),
 (107208, 307084),
 (7

In [11]:
adatas[1].var

""
GRCh38_chr1_10012_10657
GRCh38_chr1_10915_11235
GRCh38_chr1_19184_19343
GRCh38_chr1_20870_21114
GRCh38_chr1_26013_26174
...
mm10_chrY_90819540_90819845
mm10_chrY_90824173_90824421
mm10_chrY_90825257_90825564
mm10_chrY_90841656_90841860


In [12]:
for i,item in enumerate(adatas):
    adatas[i].obs['Sample'] = samples[i]
    adatas[i].obs['RL'] = sampleRL[i] 
    adatas[i].obs['Well'] = sampleWell[i] 
    adatas[i].obs['Purity'] = samplePurity[i] 
    adatas[i].obs['TagBuffer'] = sampleTagBuffer[i] 
    adatas[i].obs['TagTemp'] = sampleTagTemp[i] 
    adatas[i].obs['MErev'] = sampleMErev[i] 
    adatas[i].obs['Tn5conc'] = sampleTn5conc[i] 

In [13]:
adatas[1].obs

,Sample,RL,Well,Purity,TagBuffer,TagTemp,MErev,Tn5conc
AAACGAAAGAACTAAC-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
AAACGAAAGAAGCCTG-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
AAACGAAAGACAGCTG-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
AAACGAAAGAGAGTAG-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
AAACGAAAGAGCCACA-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
...,...,...,...,...,...,...,...,...
TTTGTGTTCTCTGACC-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
TTTGTGTTCTCTTCTC-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
TTTGTGTTCTGGCTAA-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5
TTTGTGTTCTGTGTCC-1,RL3295_39_dirty_THS_30_blockedREV_highTn5,RL3295,39,dirty,THS,30,blockedREV,highTn5


In [14]:
# check this one first (merge = same?? for ATAC?)
adatas_all = ad.concat([item for item in adatas], merge = 'same', index_unique='_')
adatas_all

AnnData object with n_obs × n_vars = 7212061 × 307084
    obs: 'Sample', 'RL', 'Well', 'Purity', 'TagBuffer', 'TagTemp', 'MErev', 'Tn5conc'

In [15]:
adatas_all.var_names

Index(['GRCh38_chr1_10012_10657', 'GRCh38_chr1_10915_11235',
       'GRCh38_chr1_19184_19343', 'GRCh38_chr1_20870_21114',
       'GRCh38_chr1_26013_26174', 'GRCh38_chr1_28636_29089',
       'GRCh38_chr1_29259_29454', 'GRCh38_chr1_29563_29908',
       'GRCh38_chr1_29949_30145', 'GRCh38_chr1_34657_35067',
       ...
       'mm10_chrY_90812243_90812718', 'mm10_chrY_90812809_90813257',
       'mm10_chrY_90818244_90818581', 'mm10_chrY_90818741_90818960',
       'mm10_chrY_90819112_90819389', 'mm10_chrY_90819540_90819845',
       'mm10_chrY_90824173_90824421', 'mm10_chrY_90825257_90825564',
       'mm10_chrY_90841656_90841860', 'mm10_chrY_90842850_90843020'],
      dtype='object', length=307084)